# Lecture 02: Spark Execution Model, DAG, & Catalyst Optimizer
This notebook contains 20 distinct, sequential code cells showcasing Spark's execution engine, Catalyst optimizer plans, narrow/wide dependencies, shuffles, and caching strategies.

### 1. Initialize SparkSession
We configure the Spark Session with local cores and optimization defaults.

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Lecture02_ExecutionModel") \
    .master("local[2]") \
    .getOrCreate()
sc = spark.sparkContext
print("Spark Session Created!")

### 2. Configure SparkContext Log Level
Setting log level to WARN to hide verbose output info.

In [ ]:
sc.setLogLevel("WARN")
print("Log level set to WARN successfully.")

### 3. Create Sample DataFrame
Generating a dataset with id and random value columns.

In [ ]:
from pyspark.sql.functions import col, rand
df = spark.range(1, 100000).withColumn("random_val", rand(seed=42))
df.show(3)

### 4. Basic Explain Plan
Printing the default physical execution plan for a filter query.

In [ ]:
filtered_df = df.filter(col("random_val") > 0.5)
filtered_df.explain()

### 5. Extended Explain: Parsed Logical Plan
Viewing the raw parsed representation of the query before any syntax/type resolution.

In [ ]:
# We can view the parsed logical plan by calling explain with extended=True
filtered_df.explain(extended=True)

### 6. Analyzed Logical Plan
Looking at the relation schema after table catalog resolution and type checks.

In [ ]:
# Analyzed Logical Plan is output as the second plan in explain(extended=True)
print("Analyzed plan contains types and resolved column identifiers.")

### 7. Optimized Logical Plan
Reviewing optimizations like Constant Folding and Filter Pushdown executed by Catalyst.

In [ ]:
# Catalyst rule optimizer outputs this plan - check the explain(extended=True) output above
print("Optimized plan eliminates redundant steps and folds constants.")

### 8. Physical Plan Operators
Identifying physical actions like HashAggregate, Exchange, and Project.

In [ ]:
# Physical plan is the final executable plan sent to JVM bytecode compiler
print("Physical plan outlines the exact task operations executed on executors.")

### 9. Lazy Evaluation Measurement
Tracing computation costs of transformations showing zero processor utilization.

In [ ]:
import time
start = time.time()
lazy_range = spark.range(1, 50000000).filter("id % 2 == 0")
print(f"Lazy range transformation setup took: {time.time() - start:.6f} seconds")

### 10. Narrow Dependency: select
Narrow operation which does not require shuffle boundaries.

In [ ]:
narrow_select = df.select("id")
print("Narrow dependency select created.")

### 11. Narrow Dependency: filter
Row filters execute locally inside partition scopes.

In [ ]:
narrow_filter = df.filter(col("id") < 1000)
print("Narrow dependency filter created.")

### 12. Wide Dependency: groupBy
Wide transformation that requires full shuffles to consolidate duplicate keys across network.

In [ ]:
wide_groupby = df.groupBy("random_val").count()
print("Wide dependency groupBy created.")

### 13. Wide Dependency: distinct
Requires shuffles to compare all row hashes globally.

In [ ]:
wide_distinct = df.select("random_val").distinct()
print("Wide dependency distinct created.")

### 14. Shuffle Tracing in Plans
Locating the 'Exchange' operator which marks network shuffles in Spark plans.

In [ ]:
wide_distinct.explain()

### 15. Caching DataFrame
Calling cache to store rows in memory with default level (MEMORY_AND_DISK).

In [ ]:
cached_df = df.filter(col("id") % 2 == 0).cache()
print("Cache requested on DataFrame.")

### 16. Persisting DataFrame with StorageLevel
Using custom level serialization options to reduce RAM memory consumption.

In [ ]:
from pyspark.storagelevel import StorageLevel
persisted_df = df.persist(StorageLevel.MEMORY_AND_DISK_SER)
print("Persist requested on DataFrame.")

### 17. Inspecting Storage Level
Verifying the deserialized/serialized state configurations.

In [ ]:
print("Persisted Storage Level:", persisted_df.storageLevel)

### 18. Double Action Benchmarking (Uncached)
Measuring duration of two count calls on uncached calculations.

In [ ]:
uncached = spark.range(1, 10000000).filter("id % 3 == 0")
start = time.time()
uncached.count()
duration_1 = time.time() - start

start = time.time()
uncached.count()
duration_2 = time.time() - start
print(f"Uncached -> Run 1: {duration_1:.4f}s, Run 2: {duration_2:.4f}s")

### 19. Double Action Benchmarking (Cached)
Measuring duration of two count calls on cached partitions showing massive speedups.

In [ ]:
cached = spark.range(1, 10000000).filter("id % 3 == 0").cache()
# Trigger cache load
cached.count()

start = time.time()
cached.count()
duration_cached_1 = time.time() - start

start = time.time()
cached.count()
duration_cached_2 = time.time() - start
print(f"Cached -> Run 1 (Loaded): {duration_cached_1:.4f}s, Run 2 (Cached): {duration_cached_2:.4f}s")

### 20. Unpersist DataFrame
Releasing cached blocks from executors memory scopes.

In [ ]:
cached.unpersist()
persisted_df.unpersist()
print("DataFrames unpersisted successfully.")

### 21. Stop Spark Session
Stopping resources cleanly.

In [ ]:
spark.stop()